# 1.0 - Preprocesamiento y Feature Engineering

**MITSUI & CO. Commodity Prediction Challenge** (Kaggle).

Objetivo: a partir de los niveles de precio crudos (`train.csv`), construir
**features con poder predictivo** para el target
`target_4 = LME_AH_Close - JPX_Gold_Standard_Futures_Close` (lag 1), y dejar
listos `X.parquet` / `y.parquet` en `data/processed/`.

Las rutas se resuelven con `src/config.py` (lee `RAW_DATA_DIR` desde `.env`).

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from src import config
from src.data.make_dataset import (
    load_raw, preprocess, engineer_features, prepare_features, build_dataset,
)
from src.data.make_dataset import LAGS, WINDOWS

config.RAW_DATA_DIR.exists(), LAGS, WINDOWS

## 1) Cargar datos crudos

In [ ]:
features, labels, pairs = load_raw()
print('features:', features.shape, '| labels:', labels.shape)
features.head()

## 2) Preprocesamiento (niveles)

Ordena por `date_id` e imputa faltantes con ffill/bfill.

In [ ]:
levels = preprocess(features)
print('NaN restantes en niveles:', int(levels.drop(columns=[config.ID_COL]).isna().sum().sum()))
levels.head()

## 3) Feature Engineering

`engineer_features` transforma los **niveles de precio** en variables con
sentido para predecir un retorno futuro (sin fuga de información):

- **Retorno** (`pct_change`) de TODAS las columnas -> estacionariedad.
- Para los **instrumentos clave** (los dos del spread) y el **spread** mismo:
  rezagos del retorno, media móvil, **volatilidad** (std móvil),
  **momentum** y **z-score** del nivel.

In [ ]:
X = engineer_features(levels)
print('features generadas:', X.shape[1] - 1)
# Algunas features del spread (las más relacionadas con el target)
[c for c in X.columns if c.startswith('spread')][:12]

In [ ]:
X.filter(regex='^spread').describe().T.head(10)

## 4) Guardar dataset procesado

`build_dataset()` = `prepare_features` (preprocess + engineer) + alineación de
labels + descarte del warmup inicial -> `data/processed/X.parquet`, `y.parquet`.

In [ ]:
x_path, y_path = build_dataset()
print('Guardado:', x_path.name, y_path.name)